In [6]:
import pandas as pd
data = {'WFAAT0': {'top_name': 'Falcon',
                   'class': 'DIV',
                   'fy': {2018:{'sub_uics': ['W1234fg', 'W4324kk', 'w8743jj']},
                          2019:{'sub_uics': ['W3427GH', 'W82TBW', 'WWWWDH0']}}}}
df = pd.DataFrame.from_dict(data, orient='index').reset_index(names='key')
df_exploded = df.explode('fy')
df_normalized = pd.json_normalize(df_exploded['fy']).set_index(df_exploded.index)
df_normalized


""
0
0


In [23]:
import pandas as pd

# Provided dictionary
data = {'WFAAT0': {'top_name': 'Falcon',
                   'class': 'DIV',
                   'fy': {2018: {'sub_uics': ['W1234fg', 'W4324kk', 'w8743jj']},
                          2019: {'sub_uics': ['W3427GH', 'W82TBW', 'WWWWDH0']}}}}

# Method 1: Using vectorized operations with json_normalize
# First, we need to restructure the data to work with json_normalize
# Convert the fy dictionary to a list of records
records = []
for key, value in data.items():
    for fy, fy_data in value['fy'].items():
        records.append({
            'key': key,
            'top_name': value['top_name'],
            'class': value['class'],
            'fy': fy,
            'sub_uics': fy_data['sub_uics']
        })

# Now use json_normalize on the restructured data
df_normalized = pd.json_normalize(records)

print("Normalized DataFrame:")
display(df_normalized)

# Now explode the sub_uics column
df_exploded = df_normalized.explode('sub_uics').reset_index(drop=True)

print("\nFinal exploded DataFrame:")
display(df_exploded)




Normalized DataFrame:


,key,top_name,class,fy,sub_uics
0,WFAAT0,Falcon,DIV,2018,"[W1234fg, W4324kk, w8743jj]"
1,WFAAT0,Falcon,DIV,2019,"[W3427GH, W82TBW, WWWWDH0]"



Final exploded DataFrame:


,key,top_name,class,fy,sub_uics
0,WFAAT0,Falcon,DIV,2018,W1234fg
1,WFAAT0,Falcon,DIV,2018,W4324kk
2,WFAAT0,Falcon,DIV,2018,w8743jj
3,WFAAT0,Falcon,DIV,2019,W3427GH
4,WFAAT0,Falcon,DIV,2019,W82TBW
5,WFAAT0,Falcon,DIV,2019,WWWWDH0


In [ ]:
# Method 2: More vectorized approach using pandas operations
import pandas as pd

# Provided dictionary
data = {'WFAAT0': {'top_name': 'Falcon',
                   'class': 'DIV',
                   'fy': {2018: {'sub_uics': ['W1234fg', 'W4324kk', 'w8743jj']},
                          2019: {'sub_uics': ['W3427GH', 'W82TBW', 'WWWWDH0']}}}}

# Convert to DataFrame first, then use vectorized operations
df_base = pd.DataFrame.from_dict(data, orient='index').reset_index()

# Create a list of dictionaries for each fy
fy_records = []
for idx, row in df_base.iterrows():
    for fy, fy_data in row['fy'].items():
        fy_records.append({
            'key': row['index'],
            'top_name': row['top_name'],
            'class': row['class'],
            'fy': fy,
            'sub_uics': fy_data['sub_uics']
        })

# Convert to DataFrame and explode
df_vectorized = pd.DataFrame(fy_records).explode('sub_uics').reset_index(drop=True)

print("Vectorized method result:")
display(df_vectorized)


Manual method result:


,key,top_name,class,fy,sub_uics
0,WFAAT0,Falcon,DIV,2018,W1234fg
1,WFAAT0,Falcon,DIV,2018,W4324kk
2,WFAAT0,Falcon,DIV,2018,w8743jj
3,WFAAT0,Falcon,DIV,2019,W3427GH
4,WFAAT0,Falcon,DIV,2019,W82TBW
5,WFAAT0,Falcon,DIV,2019,WWWWDH0


In [24]:
# Better Dictionary Structure for DataFrame Conversion
import pandas as pd

# Current structure (harder to work with):
current_structure = {
    'WFAAT0': {
        'top_name': 'Falcon',
        'class': 'DIV',
        'fy': {
            2018: {'sub_uics': ['W1234fg', 'W4324kk', 'w8743jj']},
            2019: {'sub_uics': ['W3427GH', 'W82TBW', 'WWWWDH0']}
        }
    }
}

# Better structure (easier to convert to DataFrame):
better_structure = {
    'WFAAT0': {
        'top_name': 'Falcon',
        'class': 'DIV',
        'sub_uics_by_fy': [
            {'fy': 2018, 'sub_uics': ['W1234fg', 'W4324kk', 'w8743jj']},
            {'fy': 2019, 'sub_uics': ['W3427GH', 'W82TBW', 'WWWWDH0']}
        ]
    }
}

print("Better structure:")
print(better_structure)


Better structure:
{'WFAAT0': {'top_name': 'Falcon', 'class': 'DIV', 'sub_uics_by_fy': [{'fy': 2018, 'sub_uics': ['W1234fg', 'W4324kk', 'w8743jj']}, {'fy': 2019, 'sub_uics': ['W3427GH', 'W82TBW', 'WWWWDH0']}]}}


In [ ]:
# Now with the better structure, we can use pure vectorized operations!
# This is much cleaner and more efficient

# Method 1: Using json_normalize (pure vectorized)
df_normalized = pd.json_normalize(
    better_structure, 
    record_path=['WFAAT0', 'sub_uics_by_fy'], 
    meta=[['WFAAT0', 'top_name'], ['WFAAT0', 'class']]
)

# Add the top_uic column
df_normalized['top_uic'] = 'WFAAT0'

# Explode the sub_uics to get one row per sub_uic
df_final = df_normalized.explode('sub_uics').reset_index(drop=True)

# Rename columns for clarity
df_final = df_final.rename(columns={
    'WFAAT0.top_name': 'top_name',
    'WFAAT0.class': 'class',
    'sub_uics': 'sub_uic'
})

print("Final DataFrame with better structure:")'sub_uics_by_fy']'sub'
display(df_final)


Final DataFrame with better structure:


,fy,sub_uic,top_name,class,top_uic
0,2018,W1234fg,Falcon,DIV,WFAAT0
1,2018,W4324kk,Falcon,DIV,WFAAT0
2,2018,w8743jj,Falcon,DIV,WFAAT0
3,2019,W3427GH,Falcon,DIV,WFAAT0
4,2019,W82TBW,Falcon,DIV,WFAAT0
5,2019,WWWWDH0,Falcon,DIV,WFAAT0


In [26]:
# Even better: Structure it as a list of records from the start
# This is the most DataFrame-friendly approach

best_structure = [
    {'top_uic': 'WFAAT0', 'top_name': 'Falcon', 'class': 'DIV', 'fy': 2018, 'sub_uic': 'W1234fg'},
    {'top_uic': 'WFAAT0', 'top_name': 'Falcon', 'class': 'DIV', 'fy': 2018, 'sub_uic': 'W4324kk'},
    {'top_uic': 'WFAAT0', 'top_name': 'Falcon', 'class': 'DIV', 'fy': 2018, 'sub_uic': 'w8743jj'},
    {'top_uic': 'WFAAT0', 'top_name': 'Falcon', 'class': 'DIV', 'fy': 2019, 'sub_uic': 'W3427GH'},
    {'top_uic': 'WFAAT0', 'top_name': 'Falcon', 'class': 'DIV', 'fy': 2019, 'sub_uic': 'W82TBW'},
    {'top_uic': 'WFAAT0', 'top_name': 'Falcon', 'class': 'DIV', 'fy': 2019, 'sub_uic': 'WWWWDH0'}
]

# Convert to DataFrame in one line!
df_best = pd.DataFrame(best_structure)

print("Best structure - direct to DataFrame:")
display(df_best)


Best structure - direct to DataFrame:


,top_uic,top_name,class,fy,sub_uic
0,WFAAT0,Falcon,DIV,2018,W1234fg
1,WFAAT0,Falcon,DIV,2018,W4324kk
2,WFAAT0,Falcon,DIV,2018,w8743jj
3,WFAAT0,Falcon,DIV,2019,W3427GH
4,WFAAT0,Falcon,DIV,2019,W82TBW
5,WFAAT0,Falcon,DIV,2019,WWWWDH0


In [27]:
# Function to convert your current structure to the better structure
def restructure_for_dataframe(data):
    """
    Convert the current nested structure to a list of records
    that can be easily converted to a DataFrame
    """
    records = []
    
    for top_uic, unit_data in data.items():
        top_name = unit_data['top_name']
        class_type = unit_data['class']
        
        for fy, fy_data in unit_data['fy'].items():
            for sub_uic in fy_data['sub_uics']:
                records.append({
                    'top_uic': top_uic,
                    'top_name': top_name,
                    'class': class_type,
                    'fy': fy,
                    'sub_uic': sub_uic
                })
    
    return records

# Test with your original data
original_data = {
    'WFAAT0': {
        'top_name': 'Falcon',
        'class': 'DIV',
        'fy': {
            2018: {'sub_uics': ['W1234fg', 'W4324kk', 'w8743jj']},
            2019: {'sub_uics': ['W3427GH', 'W82TBW', 'WWWWDH0']}
        }
    }
}

# Convert to records
records = restructure_for_dataframe(original_data)
display(records)

# Convert to DataFrame
df_converted = pd.DataFrame(records)

print("Converted from original structure:")
display(df_converted)


[{'top_uic': 'WFAAT0',
  'top_name': 'Falcon',
  'class': 'DIV',
  'fy': 2018,
  'sub_uic': 'W1234fg'},
 {'top_uic': 'WFAAT0',
  'top_name': 'Falcon',
  'class': 'DIV',
  'fy': 2018,
  'sub_uic': 'W4324kk'},
 {'top_uic': 'WFAAT0',
  'top_name': 'Falcon',
  'class': 'DIV',
  'fy': 2018,
  'sub_uic': 'w8743jj'},
 {'top_uic': 'WFAAT0',
  'top_name': 'Falcon',
  'class': 'DIV',
  'fy': 2019,
  'sub_uic': 'W3427GH'},
 {'top_uic': 'WFAAT0',
  'top_name': 'Falcon',
  'class': 'DIV',
  'fy': 2019,
  'sub_uic': 'W82TBW'},
 {'top_uic': 'WFAAT0',
  'top_name': 'Falcon',
  'class': 'DIV',
  'fy': 2019,
  'sub_uic': 'WWWWDH0'}]

Converted from original structure:


,top_uic,top_name,class,fy,sub_uic
0,WFAAT0,Falcon,DIV,2018,W1234fg
1,WFAAT0,Falcon,DIV,2018,W4324kk
2,WFAAT0,Falcon,DIV,2018,w8743jj
3,WFAAT0,Falcon,DIV,2019,W3427GH
4,WFAAT0,Falcon,DIV,2019,W82TBW
5,WFAAT0,Falcon,DIV,2019,WWWWDH0
